<a href="https://colab.research.google.com/github/owcaelfis6626/test/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


Sun Oct 11 11:49:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             62W /  400W |   16876MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [14]:
print("Globals in kernel:")
for k in sorted(globals().keys()):
    if not k.startswith("_"):
        v = globals()[k]
        print(f"  {k}: {type(v).__name__}")
if "history_records" in globals():
    print("\nhistory_records:", history_records)
if "total_time" in globals():
    print("total_time:", total_time)


Globals in kernel:
  BATCH_SIZE: int
  ConditionedCorrection: type
  ConfigBurgersVG2D: type
  DAMPING: float
  EXPECTED: dict
  F: module
  FourierFeatures: type
  Fraction: ABCMeta
  HEAT_MODES: int
  HeatLift: type
  In: list
  LOCAL_ITERATIONS: int
  LR: float
  MODES: int
  Out: dict
  PICARD_ITERATIONS: int
  POLYNOMIAL_NODES: int
  SEED: int
  SELECTION_DRAWS: int
  STEPS_PER_ITER: int
  ScalarPINN: type
  TEST_DRAWS: int
  TIME_NODES: int
  WIDTH: int
  WINDOW_INTERVALS: int
  amplitude: Fraction
  base_pinn: ScalarPINN
  basis: ndarray
  best_loss: float
  best_state: dict
  best_step: int
  build_frozen_realisation: function
  causal_picard_target: function
  cfg: ConfigBurgersVG2D
  cfg0: ConfigBurgersVG2D
  coef: ndarray
  cosine_coefficients: function
  dataclass: function
  dataclasses: module
  dctn: _Function
  derivative: ndarray
  derivative_matrix: function
  elapsed: float
  evaluate_field_test: function
  exit: ZMQExitAutocall
  full_basis: ndarray
  gen_sel: Gener

In [16]:
for t_val in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    pts = sample_points(cfg, 4096, gen_test, "cuda")
    pts[:, 2] = t_val
    q_val = lift.evaluate(pts)
    w_val = system(pts[:, :1], pts[:, 1:2], pts[:, 2:3])
    ref_val = interpolate_points(rec["reference"], pts, cfg)

    rms_ref = ref_val.square().mean().sqrt().item()
    rms_q_err = (q_val - ref_val).square().mean().sqrt().item()
    rms_total_err = (q_val + w_val - ref_val).square().mean().sqrt().item()

    print(f"t={t_val:.1f} | Ref RMS: {rms_ref:.4f} | q RelErr: {rms_q_err/rms_ref:.2%} | (q+w) RelErr: {rms_total_err/rms_ref:.2%}")


t=1.0 | Ref RMS: 0.1446 | q RelErr: 5.35% | (q+w) RelErr: 5.35%
t=1.1 | Ref RMS: 0.1470 | q RelErr: 38.56% | (q+w) RelErr: 38.84%
t=1.2 | Ref RMS: 0.1415 | q RelErr: 55.37% | (q+w) RelErr: 55.89%
t=1.3 | Ref RMS: 0.1336 | q RelErr: 65.15% | (q+w) RelErr: 67.24%
t=1.4 | Ref RMS: 0.1432 | q RelErr: 65.70% | (q+w) RelErr: 65.51%
t=1.5 | Ref RMS: 0.1445 | q RelErr: 69.35% | (q+w) RelErr: 70.83%
